# VLM Connector: MLP Projector, Perceiver Resampler and Image-Token Merging — Solution

**Difficulty**: 🟡 Medium

### Problem Statement

A vision-language model (LLaVA, Flamingo, Idefics, PaliGemma, Qwen-VL, …) is three parts: a **vision encoder** (a CLIP/SigLIP ViT that turns an image into a grid of patch features, e.g. $24 \times 24 = 576$ tokens of width 1024 for ViT-L/14 at 336 px), a **connector** that maps those features into the LLM's embedding space, and the **LLM**. The prompt contains `<image>` placeholder tokens, and before the first transformer layer their embeddings are replaced by the connector's outputs.

There are two families of connector:

- **MLP projector** (LLaVA-1.5's `mlp2x_gelu`): a per-token `Linear → GELU → Linear`. All 576 visual tokens reach the LLM, so no detail is lost, but every image costs 576 positions of context.
- **Perceiver Resampler** (Flamingo, IDEFICS): a fixed set of **learned latent queries** cross-attends to however many image tokens there are and always outputs `num_queries` tokens (64 in Flamingo). BLIP-2's Q-Former is a heavier relative of this.

You will build both, plus the **merge** step that splices image features into the text sequence and builds the language-modelling labels.

**Perceiver Resampler layer** (pre-LayerNorm, residual). With latents $L \in \mathbb R^{Q \times d}$ and image tokens $X \in \mathbb R^{N \times d}$:

$$\tilde X = \mathrm{LN}(X),\quad \tilde L = \mathrm{LN}(L),\quad q = \tilde L W_q,\quad [k, v] = [\tilde X;\ \tilde L]\,W_{kv}$$

$$L \leftarrow L + \mathrm{MHA}(q, k, v)\,W_o,\qquad L \leftarrow L + \mathrm{MLP}(\mathrm{LN}(L))$$

$[\tilde X; \tilde L]$ concatenates along the **token** axis: Flamingo lets the latents attend to themselves as well as to the image. Using only $\tilde X$ for keys/values is the common simpler variant, and the grader accepts both. The reference solution implements Flamingo's concatenation. A final LayerNorm is applied after the last layer. This simplified version adds **no positional encoding to the image tokens**, so the output is invariant to the order of the image tokens.

---

### Requirements

1. **`MLPProjector(vision_dim, llm_dim)`** (`nn.Module`): exactly two `nn.Linear` layers, `Linear(vision_dim, llm_dim) → GELU → Linear(llm_dim, llm_dim)`, applied to every token independently. `forward(x)`: `(B, T, vision_dim) → (B, T, llm_dim)`.
2. **`PerceiverResampler(dim, num_queries, num_heads, num_layers=1)`** (`nn.Module`):
   - `self.latents`: an `nn.Parameter` of shape `(num_queries, dim)` with a random init (e.g. `torch.randn`).
   - `num_layers` blocks, each a pre-LN multi-head cross-attention (`num_heads` heads; `dim % num_heads == 0`) with a residual connection, followed by a pre-LN MLP (`Linear(dim, 4·dim) → GELU → Linear(4·dim, dim)`) with a residual connection. LayerNorm the image tokens **and** the latents before the attention projections. Then a final `LayerNorm`.
   - `forward(x)`: `(B, N, dim) → (B, num_queries, dim)` for **any** `N`. Samples in a batch must not interact.
3. **`merge_image_features(input_ids, text_embeds, image_features, image_token_id, attention_mask=None, ignore_index=-100) -> (inputs_embeds, labels)`**
   - `input_ids`: `(B, L)` int64. Each row contains exactly `T` copies of `image_token_id` (one image per sample), at a different offset in each row (prompts differ, padding differs).
   - `text_embeds`: `(B, L, D)`, the LLM embedding lookup of `input_ids` (the rows at placeholders hold the meaningless embedding of `<image>`).
   - `image_features`: `(B, T, D)`, already projected. `image_features[b, k]` goes to the **k-th placeholder of sample b**.
   - `inputs_embeds`: `(B, L, D)` in **`text_embeds`' dtype** (cast the features: vision tower in fp32, LLM in bf16). Text positions are unchanged. Return a new tensor and don't modify any input in place. Gradients must flow to `image_features` and to `text_embeds`.
   - `labels`: `(B, L)` int64, equal to `input_ids` with placeholder positions **and** padding positions (`attention_mask == 0`, when a mask is given) set to `ignore_index`. **No shift**: the LM loss shifts later.
   - Raise **`ValueError`** if any row's number of placeholders differs from `T`.

---

### Constraints

- ✅ `F.scaled_dot_product_attention` or `nn.MultiheadAttention(batch_first=True)` are fine for the attention.
- ❌ No positional embeddings on the image tokens, and no pooling or slicing by position in the resampler.
- ❌ No Python loop over tokens **or samples** in `merge_image_features`: use a boolean mask.
- ❌ Use `raise ValueError`, not `assert`, for the count check (asserts vanish under `python -O`).

---

<details>
  <summary>💡 Hint</summary>

  - Projector: what shape does `nn.Linear` accept? Does it care about the extra `T` dimension?
  - Resampler: the latents are shared by every image. How do you give each sample in the batch its own copy without copying memory? Which tensor provides the queries, and which the keys/values?
  - A plain `.sum()` of a LayerNorm output has (almost) zero gradient. When you test gradient flow, use a random weighting.
  - Merge: `input_ids == image_token_id` is a `(B, L)` boolean mask. In what order does `x[mask]` visit the `True` positions, and how does that relate to `image_features.reshape(-1, D)`?
  - Why is the per-row count check needed even when the total count matches?
  - Padding id 0 can also be a real token. Which input tells you reliably where padding is?

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# A toy VLM: the "vision encoder" emits 16 patch features of width 64 per image; the "LLM"
# has a 1000-token vocabulary and hidden size 96. Token 999 is the <image> placeholder.
torch.manual_seed(0)
B, N_PATCH, VISION_DIM, LLM_DIM, VOCAB = 2, 16, 64, 96, 1000
IMAGE_TOKEN_ID, PAD_ID = 999, 0
vision_features = torch.randn(B, N_PATCH, VISION_DIM)   # stand-in for a CLIP/SigLIP ViT output
embed_tokens = nn.Embedding(VOCAB, LLM_DIM)             # the LLM's input embedding table


def make_batch(n_image_tokens):
    """Two prompts with the image at different offsets; sample 1 is left-padded."""
    a = [5, 17, 42, 1, 2] + [IMAGE_TOKEN_ID] * n_image_tokens + [7, 8, 9, 10]
    b = [3, 4] + [IMAGE_TOKEN_ID] * n_image_tokens + [13, 14]
    pad = len(a) - len(b)
    input_ids = torch.tensor([a, [PAD_ID] * pad + b])
    attention_mask = torch.tensor([[1] * len(a), [0] * pad + [1] * len(b)])
    return input_ids, attention_mask


ids, mask = make_batch(4)
print(ids)
print(mask)

In [ ]:
class MLPProjector(nn.Module):
    """LLaVA-1.5 vision-to-language projector: Linear -> GELU -> Linear, applied per token."""

    def __init__(self, vision_dim: int, llm_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(vision_dim, llm_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(llm_dim, llm_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, vision_dim) -> (B, T, llm_dim)"""
        # nn.Linear acts on the last dim, so every token is projected independently.
        return self.fc2(self.act(self.fc1(x)))

In [ ]:
torch.manual_seed(0)
proj = MLPProjector(VISION_DIM, LLM_DIM)
out = proj(vision_features)
assert out.shape == (B, N_PATCH, LLM_DIM), f"got {tuple(out.shape)}"
linears = [m for m in proj.modules() if isinstance(m, nn.Linear)]
assert [(l.in_features, l.out_features) for l in linears] == [(VISION_DIM, LLM_DIM), (LLM_DIM, LLM_DIM)], \
    "expected Linear(vision_dim, llm_dim) then Linear(llm_dim, llm_dim)"

# Per token: shuffling the tokens shuffles the outputs the same way.
perm = torch.randperm(N_PATCH)
assert torch.allclose(proj(vision_features[:, perm]), out[:, perm], atol=1e-6)

# Not affine: for an affine map f(x1 + x2) - f(x2) == f(x1) - f(0).
x1, x2, zero = torch.randn(1, 1, VISION_DIM), torch.randn(1, 1, VISION_DIM), torch.zeros(1, 1, VISION_DIM)
assert not torch.allclose(proj(x1 + x2) - proj(x2), proj(x1) - proj(zero), atol=1e-4), \
    "the projector is affine -- put a GELU between the two Linear layers"
print("MLPProjector tests passed")

In [ ]:
class PerceiverBlock(nn.Module):
    """One resampler layer: pre-LN cross-attention (latents -> image tokens) + pre-LN MLP."""

    def __init__(self, dim: int, num_heads: int, mlp_ratio: int = 4):
        super().__init__()
        if dim % num_heads != 0:
            raise ValueError(f"dim={dim} must be divisible by num_heads={num_heads}")
        self.num_heads = num_heads
        self.norm_media = nn.LayerNorm(dim)
        self.norm_latents = nn.LayerNorm(dim)
        self.to_q = nn.Linear(dim, dim, bias=False)
        self.to_kv = nn.Linear(dim, 2 * dim, bias=False)
        self.to_out = nn.Linear(dim, dim, bias=False)
        self.norm_ff = nn.LayerNorm(dim)
        self.ff = nn.Sequential(nn.Linear(dim, mlp_ratio * dim), nn.GELU(),
                                nn.Linear(mlp_ratio * dim, dim))

    def forward(self, x: torch.Tensor, latents: torch.Tensor) -> torch.Tensor:
        # x: (B, N, D) image tokens; latents: (B, Q, D)
        B, _, D = x.shape
        H = self.num_heads
        x_n = self.norm_media(x)
        lat_n = self.norm_latents(latents)
        q = self.to_q(lat_n)                                                   # (B, Q, D)
        # Flamingo variant: keys/values = [image tokens ; latents] along the token axis
        k, v = self.to_kv(torch.cat([x_n, lat_n], dim=1)).chunk(2, dim=-1)     # (B, N+Q, D)

        def heads(t):                                          # (B, S, D) -> (B, H, S, D/H)
            return t.view(B, t.shape[1], H, D // H).transpose(1, 2)

        attn = F.scaled_dot_product_attention(heads(q), heads(k), heads(v))   # (B, H, Q, D/H)
        attn = attn.transpose(1, 2).reshape(B, -1, D)                          # (B, Q, D)
        latents = latents + self.to_out(attn)
        return latents + self.ff(self.norm_ff(latents))


class PerceiverResampler(nn.Module):
    """
    Flamingo-style Perceiver Resampler: num_queries learned latent vectors cross-attend to a
    variable number of image tokens and come out as exactly num_queries visual tokens.
    """

    def __init__(self, dim: int, num_queries: int, num_heads: int, num_layers: int = 1):
        super().__init__()
        self.latents = nn.Parameter(torch.randn(num_queries, dim))
        self.layers = nn.ModuleList([PerceiverBlock(dim, num_heads) for _ in range(num_layers)])
        self.norm_out = nn.LayerNorm(dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, N, dim) image tokens, any N -> (B, num_queries, dim)"""
        # expand() is a view: every sample starts from the same learned queries, and the
        # gradients from all samples accumulate into self.latents.
        latents = self.latents.unsqueeze(0).expand(x.shape[0], -1, -1)        # (B, Q, D)
        for layer in self.layers:
            latents = layer(x, latents)
        return self.norm_out(latents)

In [ ]:
torch.manual_seed(0)
NUM_QUERIES = 8
resampler = PerceiverResampler(LLM_DIM, NUM_QUERIES, num_heads=4, num_layers=2)
for n in (1, 16, 100):
    out = resampler(torch.randn(B, n, LLM_DIM))
    assert out.shape == (B, NUM_QUERIES, LLM_DIM), f"{n} image tokens -> {tuple(out.shape)}"

tokens = proj(vision_features).detach()                       # (B, 16, LLM_DIM)
with torch.no_grad():
    perm = torch.randperm(N_PATCH)
    assert torch.allclose(resampler(tokens), resampler(tokens[:, perm]), atol=1e-5), \
        "the output depends on the order of the image tokens"
    assert torch.allclose(resampler(tokens)[1:], resampler(tokens[1:]), atol=1e-5), \
        "samples in a batch must not interact"

assert isinstance(resampler.latents, nn.Parameter) and resampler.latents.shape == (NUM_QUERIES, LLM_DIM)
out = resampler(tokens)
(out * torch.randn_like(out)).sum().backward()   # weighted: a LayerNorm output's plain sum has ~0 grad
assert resampler.latents.grad is not None and resampler.latents.grad.abs().sum() > 0, \
    "no gradient reached the latent queries"
n_params = sum(p.numel() for p in resampler.parameters())
print(f"{N_PATCH} image tokens -> {NUM_QUERIES} visual tokens; resampler has {n_params:,} parameters")
print("PerceiverResampler tests passed")

In [ ]:
def merge_image_features(input_ids: torch.Tensor, text_embeds: torch.Tensor,
                         image_features: torch.Tensor, image_token_id: int,
                         attention_mask: torch.Tensor | None = None,
                         ignore_index: int = -100):
    """
    Splice image features into the text embedding sequence at the <image> placeholders.

    Args:
        input_ids:      (B, L) int64; every row contains exactly T copies of image_token_id.
        text_embeds:    (B, L, D) embeddings of input_ids (placeholder rows get overwritten).
        image_features: (B, T, D) projected image tokens; image_features[b, k] goes to the
                        k-th placeholder of sample b.
        image_token_id: id of the <image> placeholder token.
        attention_mask: optional (B, L); 1 = real token, 0 = padding.
        ignore_index:   label value the LM loss skips.

    Returns:
        inputs_embeds: (B, L, D) in text_embeds' dtype -- a new tensor, inputs untouched.
        labels:        (B, L) int64: input_ids with placeholder and padding positions set to
                       ignore_index. NOT shifted.

    Raises:
        ValueError: if any row's number of placeholders differs from T.
    """
    B, L, D = text_embeds.shape
    if image_features.dim() != 3 or image_features.shape[0] != B or image_features.shape[2] != D:
        raise ValueError(f"image_features should be ({B}, T, {D}), got {tuple(image_features.shape)}")
    T = image_features.shape[1]

    is_image = input_ids == image_token_id                       # (B, L) bool
    counts = is_image.sum(dim=1)                                 # (B,) placeholders per row
    if (counts != T).any():   # per row: a matching TOTAL could still misalign samples
        raise ValueError(f"each sequence needs exactly {T} image placeholders "
                         f"(token {image_token_id}); got per-sample counts {counts.tolist()}")

    inputs_embeds = text_embeds.clone()                          # never write into the input
    # x[mask] visits True positions in row-major order: sample 0's placeholders left to
    # right, then sample 1's, ... -- the same order as image_features.reshape(-1, D).
    inputs_embeds[is_image] = image_features.reshape(-1, D).to(inputs_embeds.dtype)

    labels = input_ids.masked_fill(is_image, ignore_index)       # no target at image slots
    if attention_mask is not None:
        labels = labels.masked_fill(attention_mask == 0, ignore_index)   # nor at padding
    return inputs_embeds, labels

In [ ]:
T = 4
ids, mask = make_batch(T)
text_embeds = embed_tokens(ids).detach()
# Distinguishable features: every entry of feature k of sample b equals 4*b + k + 1.
feats = torch.arange(1, B * T + 1, dtype=torch.float32).view(B, T, 1).expand(B, T, LLM_DIM)
embeds, labels = merge_image_features(ids, text_embeds, feats, IMAGE_TOKEN_ID, attention_mask=mask)

is_img = ids == IMAGE_TOKEN_ID
assert embeds.shape == text_embeds.shape and embeds.dtype == text_embeds.dtype
for b in range(B):
    pos = is_img[b].nonzero().flatten()
    assert torch.equal(embeds[b, pos, 0], feats[b, :, 0]), \
        f"sample {b}: expected features {feats[b, :, 0].tolist()} at its placeholders, got {embeds[b, pos, 0].tolist()}"
assert torch.equal(embeds[~is_img], text_embeds[~is_img]), "text positions must be untouched"
assert torch.equal(labels, ids.masked_fill(is_img | (mask == 0), -100)), f"wrong labels:\n{labels}"

bad = ids.clone()
bad[0, is_img[0].nonzero()[0]] = 5                 # sample 0 now has T - 1 placeholders
try:
    merge_image_features(bad, text_embeds, feats, IMAGE_TOKEN_ID, attention_mask=mask)
except ValueError as e:
    print("count mismatch raises ValueError:", e)
else:
    raise AssertionError("a placeholder-count mismatch must raise ValueError")
print("merge_image_features tests passed")

In [ ]:
# End to end: ViT features -> connector -> merge -> toy LM head -> next-token loss.
torch.manual_seed(0)
lm_head = nn.Linear(LLM_DIM, VOCAB)

def lm_loss(embeds, labels):
    logits = lm_head(embeds)                                  # (B, L, VOCAB); a real LLM goes here
    # The shift happens HERE: position t predicts token t + 1.
    return F.cross_entropy(logits[:, :-1].reshape(-1, VOCAB), labels[:, 1:].reshape(-1),
                           ignore_index=-100)

for name, connector, n_vis in (("MLP projector", MLPProjector(VISION_DIM, LLM_DIM), N_PATCH),
                               ("projector + resampler",
                                nn.Sequential(MLPProjector(VISION_DIM, LLM_DIM),
                                              PerceiverResampler(LLM_DIM, 8, num_heads=4)), 8)):
    ids, mask = make_batch(n_vis)
    embeds, labels = merge_image_features(ids, embed_tokens(ids), connector(vision_features),
                                          IMAGE_TOKEN_ID, attention_mask=mask)
    loss = lm_loss(embeds, labels)
    loss.backward()
    assert torch.isfinite(loss)
    assert all(p.grad is not None for p in connector.parameters()), "the connector got no gradient"
    n_sup = int((labels[:, 1:] != -100).sum())
    print(f"{name:22s}: sequence length {ids.shape[1]:2d}, {n_vis:2d} visual tokens/image, "
          f"{n_sup} supervised positions, loss {loss.item():.3f}")
print("end-to-end test passed")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("vlm-connector", MLPProjector=MLPProjector, PerceiverResampler=PerceiverResampler,
      merge_image_features=merge_image_features)

### Why it works

- **Projector.** `nn.Linear` acts on the last dimension, so `(B, T, d_v) → (B, T, d_l)` maps each visual token on its own. The GELU makes the mapping non-linear, which is the improvement LLaVA-1.5 made over LLaVA-1's single linear layer. The projector learns to put visual features where the LLM's word embeddings live.
- **Resampler.** Attention is a weighted average over keys/values, with weights computed from query–key dot products. The **queries** set the number of outputs, so `num_queries` learned latents always give `num_queries` tokens whatever `N` is. Without positional encodings the keys/values form a **set**: permuting the image tokens permutes the softmax weights and the values together, and the output does not change. The LayerNorm on the image tokens makes the block insensitive to the scale of the vision features, and the residual + MLP structure is an ordinary pre-LN transformer block whose "self" is the latent array.
- **Merge.** `x[mask]` enumerates `True` positions in C (row-major) order. Because each row holds exactly `T` placeholders, that order is `(b=0, k=0..T-1), (b=1, k=0..T-1), …`, which is the order of `image_features.reshape(-1, D)`. One masked assignment therefore puts the k-th feature of sample b at the k-th placeholder of sample b, wherever the run starts. Autograd handles the in-place `index_put_` on the clone: overwritten positions pass zero gradient to `text_embeds`, and each feature receives exactly its own position's gradient.
- **Labels.** At placeholder positions the "token" is not something the model should learn to emit, and at padding there is nothing to predict, so both get `ignore_index` and `F.cross_entropy` skips them. The shift by one happens inside the LM loss (`logits[:, :-1]` vs `labels[:, 1:]`), which is the Hugging Face convention.

### Complexity

- Projector: $O(T(d_v d_l + d_l^2))$, no interaction between tokens.
- Resampler: per layer $O(Q(N+Q)d)$ for attention plus $O((N+Q)d^2)$ for the projections. It is **linear** in the number of image tokens $N$, and the LLM then sees $Q \ll N$ tokens. Because LLM attention is quadratic in sequence length, this matters a lot for video and multi-image inputs.
- Merge: $O(BLD)$ memory traffic (one clone and one scatter), no Python loops.

### Common bugs

- Checking only the **total** placeholder count. Two samples with $T+1$ and $T-1$ placeholders silently shift one feature into the wrong sample.
- Flattening the features in the wrong order (`image_features.transpose(0, 1).reshape(-1, D)`), or assuming every sample's image starts at the same position.
- Writing into `text_embeds` in place. That corrupts the caller's tensor, and fails outright when `text_embeds` is a leaf that requires grad.
- Finding padding with `input_ids == pad_id`, which also masks real occurrences of that id. Use the attention mask.
- Shifting the labels here **and** in the loss (double shift), or not masking image positions, so the model is trained to predict `<image>` tokens.
- Forgetting the dtype cast (`index_put_` refuses fp32 into bf16), or `.detach()`-ing the features, so the projector never trains.
- Resampler: `nn.MultiheadAttention` without `batch_first=True` attends across the batch. Adding a positional embedding sized to a fixed `N` breaks variable-resolution input. A plain `.sum()` loss after the final LayerNorm makes gradient checks look broken.

### Interview follow-ups

1. **Projector vs resampler vs Q-Former: what are the trade-offs?** The MLP keeps all $N$ tokens (576 per image for LLaVA-1.5), so fine detail survives, which helps OCR, charts and counting, but context cost grows with resolution and image count. A resampler or Q-Former compresses to a fixed 32–64 tokens: cheap and good for video and multi-image input, but lossy, and it struggles with dense text. BLIP-2's Q-Former also needs its own pre-training stage. Many recent models keep an MLP but reduce tokens by merging neighbours first (e.g. 2×2 patch merging or pixel shuffle before the MLP in Qwen2-VL and InternVL).
2. **How does dynamic-resolution tiling (LLaVA-NeXT "AnyRes") work?** Choose, from a set of candidate grids, the one whose aspect ratio and resolution best fit the image. Resize and pad the image to it, split it into encoder-sized tiles (336²), encode each tile separately, and add a downsampled global view. The tokens are then concatenated, with LLaVA-NeXT inserting a learned newline token after each row. Token count grows with resolution (up to $5 \times 576 = 2880$ for a 2×2 grid plus the global view). Native-resolution ViTs with 2-D RoPE (Qwen2-VL) avoid tiling altogether.
3. **What is frozen in each training stage?** LLaVA: stage 1 (feature alignment on ~558K caption pairs) freezes the ViT and the LLM and trains only the projector. Stage 2 (visual instruction tuning) keeps the ViT frozen and trains the projector and the LLM (fully or with LoRA). Flamingo freezes both the vision encoder and the LM and trains the resampler and the new gated cross-attention layers. BLIP-2 trains the Q-Former against a frozen image encoder, then connects it to a frozen LLM.
4. **Why are image tokens excluded from the LM loss?** They are continuous inputs, not vocabulary tokens, so there is no meaningful target id at those positions. Supervising them would train the model to emit `<image>` placeholders and waste capacity. For the same reason, instruction tuning usually also masks the system and user turns, so only the assistant's answer tokens are supervised.
5. **Where do the visual tokens go: input sequence or cross-attention?** LLaVA-style models concatenate them into the input sequence (simple, and every layer sees them, but they use context). Flamingo-style models keep the text sequence short and inject vision through gated cross-attention layers inserted between frozen LM blocks, with $\tanh$ gates initialised at 0 so the frozen LM starts out unchanged. Cross-attention scales better to many images but needs architectural changes to the LM.
6. **How would you batch images of different sizes, or samples with 0 or 2 images?** Count placeholders per image instead of per sample. Flatten all images' features to `(total_image_tokens, D)` and scatter them with the same mask (`masked_scatter`), checking that `mask.sum()` equals the number of feature rows. Per-image token counts can differ under AnyRes, so carry an explicit list of sizes rather than a dense `(B, T, D)` tensor.